In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
from scipy.io import mmread
from scipy.sparse import csr_matrix
from anndata import AnnData

# List of study folders
folders = [
    "Chen_2024",
    "Chow_2023",
    "Liu_2022",
    "Liu_2025",
    "Zheng_2021"
]

data_dir = "/Users/wsun/research/CAT/"
meta_dir = "/Users/wsun/research/CAT/data"

## Read in data

In [2]:
# Step 1: Read and collect gene names
gene_lists = {}
adata_dict = {}

for folder in folders:
    print(f"Reading {folder}...")

    # Read files
    matrix = mmread(f"{data_dir}{folder}/{folder}_CD4/matrix.mtx.gz").tocsr()
    genes = pd.read_csv(f"{data_dir}{folder}/{folder}_CD4/genes.tsv", header=None, sep="\t")[0]
    barcodes = pd.read_csv(f"{data_dir}{folder}/{folder}_CD4/barcodes.tsv", header=None)[0]

    # Store gene list for intersection
    gene_lists[folder] = genes

    # Create AnnData object
    adata = AnnData(X=matrix)
    adata.var_names = genes
    adata.obs_names = barcodes
    adata.obs["study"] = folder

    adata_dict[folder] = adata

Reading Chen_2024...
Reading Chow_2023...
Reading Liu_2022...
Reading Liu_2025...
Reading Zheng_2021...


In [3]:
for k, v in adata_dict.items():
    print(f"{k}: {v.shape}")

Chen_2024: (210900, 16323)
Chow_2023: (93055, 16323)
Liu_2022: (58600, 16323)
Liu_2025: (307895, 16323)
Zheng_2021: (33435, 16323)


In [4]:
# Step 2: Intersect genes
common_genes = set.intersection(*(set(g) for g in gene_lists.values()))
common_genes = sorted(list(common_genes))  # Ensure consistent order
print(f"Found {len(common_genes)} common genes.")

Found 16323 common genes.


In [5]:
# Step 3: Subset each AnnData to common genes
for k in adata_dict:
    adata_dict[k] = adata_dict[k][:, common_genes]

# Step 4: Concatenate into one AnnData object
adata_combined = adata_dict[folders[0]].concatenate(
    *[adata_dict[f] for f in folders[1:]],
    batch_key="study",
    batch_categories=folders
)

print(f"Combined AnnData shape: {adata_combined.shape}")

/var/folders/fk/gk1pvdpx7fz79tx26t0rjc7c0000gp/T/ipykernel_26831/3122805540.py:6: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_combined = adata_dict[folders[0]].concatenate(


Combined AnnData shape: (703885, 16323)


In [6]:
sc.pp.calculate_qc_metrics(
    adata_combined,
    percent_top=None,      # you can set e.g., [50, 100, 200] if you want top-n genes metrics
    log1p=False,           # don't log-transform counts
    inplace=True           # store results directly in adata.obs / adata.var
)
print(adata_combined.obs.head())


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


                                          study  n_genes_by_counts  \
CRC01-N-I_AAACCTGAGGGTGTGT-Chen_2024  Chen_2024               1714   
CRC01-N-I_AAACCTGGTGACGGTA-Chen_2024  Chen_2024               1663   
CRC01-N-I_AAACGGGGTCGTCTTC-Chen_2024  Chen_2024               1299   
CRC01-N-I_AAAGATGCATTTCACT-Chen_2024  Chen_2024               1542   
CRC01-N-I_AAAGCAATCTACTCAT-Chen_2024  Chen_2024               2141   

                                      total_counts  
CRC01-N-I_AAACCTGAGGGTGTGT-Chen_2024        4285.0  
CRC01-N-I_AAACCTGGTGACGGTA-Chen_2024        4760.0  
CRC01-N-I_AAACGGGGTCGTCTTC-Chen_2024        2479.0  
CRC01-N-I_AAAGATGCATTTCACT-Chen_2024        5079.0  
CRC01-N-I_AAAGCAATCTACTCAT-Chen_2024        6964.0  


In [7]:
X = adata_combined.X
print(f"adata.X shape: {X.shape}")
print(X[100:110,100:105])

# Get the number of cells
n_cells = X.shape[0]

# Count how many cells express each gene (non-zero)
gene_expr_counts = (X > 0).sum(axis=0)
gene_expr_counts = np.asarray(gene_expr_counts).flatten()

# Compute fraction of expressing cells per gene
gene_expr_frac = gene_expr_counts / n_cells

# Convert to pandas Series for convenience
gene_frac_series = pd.Series(gene_expr_frac, index=adata_combined.var_names)

# Define thresholds
thresholds = [0.01, 0.02, 0.03, 0.05]

# Count how many genes pass each threshold
for t in thresholds:
    count = (gene_frac_series >= t).sum()
    print(f"Genes expressed in ≥{int(t * 100)}% of cells: {count}")


adata.X shape: (703885, 16323)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (10, 5)>
  Coords	Values
  (0, 2)	1.0
  (0, 1)	1.0
  (1, 1)	2.0
  (2, 1)	3.0
  (2, 4)	1.0
  (3, 2)	1.0
  (3, 1)	1.0
  (4, 1)	2.0
  (9, 1)	1.0
Genes expressed in ≥1% of cells: 9507
Genes expressed in ≥2% of cells: 8465
Genes expressed in ≥3% of cells: 7592
Genes expressed in ≥5% of cells: 6241


In [8]:
keep_genes = gene_expr_frac >= 0.05
genes_to_keep = adata_combined.var_names[keep_genes]

adata_combined = adata_combined[:, genes_to_keep].copy()
print(f"Filtered to {adata_combined.n_vars} genes expressed in ≥5% of cells.")

Filtered to 6241 genes expressed in ≥5% of cells.


In [9]:
file_path = os.path.join(meta_dir, "CD4_updated.tsv.gz")
cell_info = pd.read_csv(file_path, sep="\t", compression="gzip")

print(cell_info.shape)
print(cell_info.head())

(390785, 41)
          TRB_cdr3 TRB_v_gene           TRA_cdr3    TRA_v_gene  \
0  CA*MRGFIMATPSVR     TRBV30     CADYSGGGADGLTF      TRAV13-1   
1    CAAAATNNNEQFF   TRBV10-3        CAGSNTDKLIF    TRAV29/DV5   
2     CAAAGAGTEAFF    TRBV6-5   CALSEARAGGSYIPTF        TRAV19   
3   CAAAGGPKSGELFF    TRBV7-8         CVAEGHDMRF      TRAV12-1   
4    CAAAGQDNSPLHF    TRBV3-1  CAYRGGNSGGSNYKLTF  TRAV38-2/DV8   

                                               clone  \
0     TRAV13-1_CADYSGGGADGLTF_TRBV30_CA*MRGFIMATPSVR   
1      TRAV29/DV5_CAGSNTDKLIF_TRBV10-3_CAAAATNNNEQFF   
2       TRAV19_CALSEARAGGSYIPTF_TRBV6-5_CAAAGAGTEAFF   
3         TRAV12-1_CVAEGHDMRF_TRBV7-8_CAAAGGPKSGELFF   
4  TRAV38-2/DV8_CAYRGGNSGGSNYKLTF_TRBV3-1_CAAAGQD...   

                       cell_id      study cancer_type  \
0     AGTGGGATCTGACCTC.58-THCA  Zheng2021        THCA   
1   CRC20-B-I_AACCGCGTCTGATACG   Chen2024        COAD   
2  CRC04-B-II_ATTTCTGCAGTAAGCG   Chen2024        COAD   
3   CGCTTCAAGGCAAAGA-1_PE

In [10]:
original_study_counts = cell_info['study'].value_counts()
print("Before modification:")
print(original_study_counts)

# Modify study column: insert "_" before the 4-digit year
cell_info['study'] = cell_info['study'].str.replace(r'(\D)(\d{4})$', r'\1_\2', regex=True)

# Tabulate modified study values
modified_study_counts = cell_info['study'].value_counts()
print("\nAfter modification:")
print(modified_study_counts)

cell_info['cell_id_long'] = cell_info['cell_id'] + "-" + cell_info['study']


Before modification:
study
Liu2025      157105
Chen2024     109721
Chow2023      68222
Zheng2021     33435
Liu2022       22302
Name: count, dtype: int64

After modification:
study
Liu_2025      157105
Chen_2024     109721
Chow_2023      68222
Zheng_2021     33435
Liu_2022       22302
Name: count, dtype: int64


In [11]:
adata_cells = set(adata_combined.obs_names)
meta_cells = set(cell_info['cell_id_long'])

print(list(adata_cells)[:10])
print(list(meta_cells)[:10])

# Cells in metadata but not in AnnData
missing_in_adata = meta_cells - adata_cells
# Cells in AnnData but not in metadata
missing_in_meta = adata_cells - meta_cells

print(f"\nNumber of matching cell IDs: {len(meta_cells & adata_cells)}")
print(f"Number of cell IDs in metadata not in adata_combined: {len(missing_in_adata)}")
print(f"Number of cell IDs in adata_combined not in metadata: {len(missing_in_meta)}")

ordered_meta_cells = cell_info['cell_id_long'].tolist()

adata_combined = adata_combined[ordered_meta_cells, :].copy()
assert list(adata_combined.obs_names) == ordered_meta_cells, "Ordering mismatch!"

print(f"adata_combined now contains {adata_combined.n_obs} cells.")


['CRC20-N-III_GAGGTGATCCGCATCT-Chen_2024', 'CRC24-B-II_AGCTCCTGTGCAGACA-Chen_2024', 'CRC08-B-I_GGACAAGTCCTTCAAT-Chen_2024', 'ATCCACCAGGGCATGT.44-RC-Zheng_2021', 'P464-CCAATCCCAGCTGTGC-1-Liu_2025', 'P45-CTGATAGCACGGTAGA-1-Liu_2025', 'P21.ut.CTGAAGTAGCGCTTAT-1-Liu_2022', 'CRC18-B-I_CCTACACCAACTGGCC-Chen_2024', 'P13.ut.TACTTACAGCTGGAAC-1-Liu_2022', 'ACCCACTCATCCTTGC.58-THCA-Zheng_2021']
['TCGAGGCTCTACCTGC-1_PEM10C1-Chow_2023', 'P66-ACCAGTATCAGCCTAA-1-Liu_2025', 'CACCACTGTCATCGGC-1_PEM15C5-Chow_2023', 'CRC24-B-II_AGCTCCTGTGCAGACA-Chen_2024', 'ATCCACCAGGGCATGT.44-RC-Zheng_2021', 'P32-TGCACCTTCCTCTAGC-1-Liu_2025', 'P45-CTGATAGCACGGTAGA-1-Liu_2025', 'TCAATCTCAGCTTAAC.30-MM-Zheng_2021', 'P59-GATCGCGAGAGCTGCA-1-Liu_2025', 'CRC01-T-IV_TGAGAGGCAGATGAGC-Chen_2024']

Number of matching cell IDs: 390785
Number of cell IDs in metadata not in adata_combined: 0
Number of cell IDs in adata_combined not in metadata: 313100
adata_combined now contains 390785 cells.


In [12]:
cell_info = cell_info.set_index("cell_id_long")
pd.set_option("display.max_columns", None)  # Show all columns
print(cell_info.iloc[:2, :])

                                             TRB_cdr3 TRB_v_gene  \
cell_id_long                                                       
AGTGGGATCTGACCTC.58-THCA-Zheng_2021   CA*MRGFIMATPSVR     TRBV30   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024    CAAAATNNNEQFF   TRBV10-3   

                                            TRA_cdr3  TRA_v_gene  \
cell_id_long                                                       
AGTGGGATCTGACCTC.58-THCA-Zheng_2021   CADYSGGGADGLTF    TRAV13-1   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024     CAGSNTDKLIF  TRAV29/DV5   

                                                                               clone  \
cell_id_long                                                                           
AGTGGGATCTGACCTC.58-THCA-Zheng_2021   TRAV13-1_CADYSGGGADGLTF_TRBV30_CA*MRGFIMATPSVR   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024   TRAV29/DV5_CAGSNTDKLIF_TRBV10-3_CAAAATNNNEQFF   

                                                         cell_id       study  \
cell_id_long         

In [13]:
print(adata_combined.obs.iloc[:2, :])

                                           study  n_genes_by_counts  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021   Zheng_2021               1346   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024   Chen_2024               1029   

                                      total_counts  
AGTGGGATCTGACCTC.58-THCA-Zheng_2021         3215.0  
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024        2930.0  


In [14]:
adata_combined.obs.rename(columns={"study": "study2"}, inplace=True)
adata_combined.obs = adata_combined.obs.join(cell_info)
print(adata_combined.obs.head())
adata_combined

                                           study2  n_genes_by_counts  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021    Zheng_2021               1346   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024    Chen_2024               1029   
CRC04-B-II_ATTTCTGCAGTAAGCG-Chen_2024   Chen_2024               1236   
CGCTTCAAGGCAAAGA-1_PEM15C5-Chow_2023    Chow_2023                971   
P47-TCAGCTCGTTCAGCGC-1-Liu_2025          Liu_2025               1041   

                                       total_counts         TRB_cdr3  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021          3215.0  CA*MRGFIMATPSVR   
CRC20-B-I_AACCGCGTCTGATACG-Chen_2024         2930.0    CAAAATNNNEQFF   
CRC04-B-II_ATTTCTGCAGTAAGCG-Chen_2024        3943.0     CAAAGAGTEAFF   
CGCTTCAAGGCAAAGA-1_PEM15C5-Chow_2023         2697.0   CAAAGGPKSGELFF   
P47-TCAGCTCGTTCAGCGC-1-Liu_2025              1501.0    CAAAGQDNSPLHF   

                                      TRB_v_gene           TRA_cdr3  \
AGTGGGATCTGACCTC.58-THCA-Zheng_2021       TRBV30     CADYSGGGAD

AnnData object with n_obs × n_vars = 390785 × 6241
    obs: 'study2', 'n_genes_by_counts', 'total_counts', 'TRB_cdr3', 'TRB_v_gene', 'TRA_cdr3', 'TRA_v_gene', 'clone', 'cell_id', 'study', 'cancer_type', 'Patient', 'Sample', 'Treatment', 'Tissue', 'study_specific_CR_per_cell', 'study_specific_CR_by_cluster', 'barcode', 'TRA_j_gene', 'TRB_j_gene', 'TRA_nUMI', 'TRB_nUMI', 'CD4_Caushi_Tfh2_66g', 'CD4_Lowery_neg_37g', 'CD4_Lowery_pos_40g', 'CD4_Oh_CXCL13_50g', 'CD4_ave_Hanada_pos_9g', 'CD4_ave_Hanada_neg_4g', 'study_clone_id', 'study_clone_number', 'pos_score_CD4', 'neg_score_CD4', 'cancer_reactive_per_cell', 'cancer_reactive', 'total_cells_patient', 'clone_number_per_patient', 'clone_number_total', 'clone_n_patient', 'clone_number_per_patient_median', 'clone_freq_per_patient', 'TRA_antigen', 'TRB_antigen', 'non_human_antigen', 'label'
    var: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [15]:
adata_combined.write(os.path.join(meta_dir, "CD4_combined_filtered.h5ad"))

print(f"Saved filtered AnnData to: {os.path.join(meta_dir, 'CD4_combined_filtered.h5ad')}")


Saved filtered AnnData to: /Users/wsun/research/CAT/data/CD4_combined_filtered.h5ad
